# Chunking Strategy Experiments

Goal: chunk real scraped docs for Milvus ingest (or compare strategies).

**Flow**
1. Load docs from GitHub + join CSV metadata (per document). `SELECTED_FILES = []` = all files
2. Run §6 prepare, then **exactly one** strategy — writes `chunked_docs` + flat `chunks`
3. Build `selected_chunks.json` → copy to Drive → continue in `schema_design_notebook`


### 1. Install dependencies

In [1]:
!pip install --quiet groq llama-index-core llama-index-embeddings-huggingface tiktoken nltk sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 59.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 42.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 6.4 MB/s eta 0:00:00


### 2. Set up Groq

In [ ]:
import os
import numpy as np
from groq import Groq
from llama_index.core import Document
from llama_index.core.node_parser import TokenTextSplitter, SentenceSplitter

# Paste your Groq API key here (or set it as an env var before running).
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

client = Groq(api_key=GROQ_API_KEY)
GROQ_MODEL = "llama-3.3-70b-versatile"

# Quick check that Groq works
resp = client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": "Reply with just the word: ready"}],
)
print("Groq says:", resp.choices[0].message.content.strip())

Groq says: ready


### 3. Load docs from GitHub

Always fetches from the public repo (works on Colab and locally).

Leave `SELECTED_FILES` empty to load **everything** in `data/markdown`, or list one/two filenames for faster experiments.

Next cell joins these files to `raw_data.csv` by `doc_id`.


In [11]:
from pathlib import Path
import shutil
import subprocess
import urllib.request

GITHUB_REPO = "harshkumar1/website-scrapper"
GITHUB_BRANCH = "main"
GITHUB_DOCS_SUBDIR = "data/markdown"

# Leave empty to use ALL text files.
# Or list one/two filenames for faster experiments:
# SELECTED_FILES = ["0167d8f6-b000-4304-bbfe-5c981ac1aa55.md"]  # just one
# SELECTED_FILES = ["0167d8f6-b000-4304-bbfe-5c981ac1aa55.md", "2248c409-bc5b-4933-931a-b42c15d2e367.md"]  # two
SELECTED_FILES = []  # all matching files

TEXT_EXTENSIONS = {".md"}


def ensure_docs_dir() -> Path:
    if SELECTED_FILES:
        cache_dir = Path("docs_from_github")
        cache_dir.mkdir(parents=True, exist_ok=True)
        for name in SELECTED_FILES:
            dest = cache_dir / name
            if dest.exists():
                continue
            url = (
                f"https://raw.githubusercontent.com/{GITHUB_REPO}/"
                f"{GITHUB_BRANCH}/{GITHUB_DOCS_SUBDIR}/{name}"
            )
            print(f"Downloading from GitHub: {name}")
            urllib.request.urlretrieve(url, dest)
        return cache_dir.resolve()

    # All files: shallow sparse clone of just data/markdown
    repo_root = Path("docs_from_github_repo")
    docs_subdir = repo_root / GITHUB_DOCS_SUBDIR
    if not docs_subdir.is_dir():
        if repo_root.exists():
            shutil.rmtree(repo_root)
        print("Sparse-cloning docs from GitHub (first run only)...")
        subprocess.run(
            [
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                f"https://github.com/{GITHUB_REPO}.git", str(repo_root),
            ],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repo_root), "sparse-checkout", "set", GITHUB_DOCS_SUBDIR],
            check=True,
        )
    return docs_subdir.resolve()


docs_dir = ensure_docs_dir()
if not docs_dir.is_dir():
    raise FileNotFoundError(f"Docs folder not found: {docs_dir}")

if SELECTED_FILES:
    selected_paths = []
    for name in SELECTED_FILES:
        path = docs_dir / name
        if not path.is_file():
            raise FileNotFoundError(f"File not found in {docs_dir}: {name}")
        selected_paths.append(path)
else:
    selected_paths = sorted(
        p for p in docs_dir.iterdir()
        if p.is_file() and p.suffix.lower() in TEXT_EXTENSIONS
    )

if not selected_paths:
    raise FileNotFoundError(f"No text files found in {docs_dir}")

print(f"Docs folder: {docs_dir}")
print(f"Using {len(selected_paths)} file(s):")
for path in selected_paths:
    print(f"  - {path.name}")


Docs folder: /content/docs_from_github_repo/data/markdown
Using 368 file(s):
  - 0154c3ea-32a1-4eac-a40d-cc394c506a97.md
  - 0167d8f6-b000-4304-bbfe-5c981ac1aa55.md
  - 01795032-f63f-421e-8387-ddbc5df246c4.md
  - 02359dc4-8ff4-4084-a694-8a516904e100.md
  - 027d87b6-6cff-4e78-ad8f-e51761a424e6.md
  - 035de21d-4241-4bb2-933c-4d5b0fbaae7c.md
  - 03944a6e-edf7-477e-883e-7c9753b2685e.md
  - 03a71673-a30f-42cb-a3e3-936f08d156d0.md
  - 03ca845f-5a67-4c43-98f5-e4e7220a1d86.md
  - 04bcd169-ce6a-4836-9dca-51f817dc1d70.md
  - 05b8c7dc-85bf-4262-a7f7-a8ef2434f666.md
  - 05cb9a8a-cec4-4d84-961b-0d170560a0ea.md
  - 06393c5f-3f0d-4365-827d-237c07f470ec.md
  - 0647780e-3957-4813-b349-be510031a2c6.md
  - 07612a02-2ebe-4d44-9045-db22b1077d8e.md
  - 077739c9-0e0b-49b0-bad2-f7cc9fa3be20.md
  - 077e48d9-daec-4a43-bec0-079d65d68207.md
  - 07fb6de9-05d8-40ac-a633-704faef76ae7.md
  - 081f3adf-0ed0-44b0-ad81-e6f3e1493bfd.md
  - 092ca5f0-24d5-4b0f-807e-e33866237f3b.md
  - 09307d95-07ad-408c-91fd-7d420cef9ddd.md

### 4. Load CSV + join markdown (per document)

Ingest-ready load (do this **before** experiments):
1. Download `data/raw_data.csv` from GitHub
2. Join each selected `.md` to its CSV row via `doc_id` (= filename without `.md`)
3. Keep **one entry per document** in `documents` — do not merge into one blob

`SELECTED_FILES` in §3 controls how many docs: 1, 2, … or `[]` for all.


In [12]:
import csv
import urllib.request
from pathlib import Path

GITHUB_DATA_SUBDIR = "data"
CSV_NAME = "raw_data.csv"

csv_path = Path("docs_from_github") / CSV_NAME
csv_path.parent.mkdir(parents=True, exist_ok=True)
if not csv_path.exists():
    csv_url = (
        f"https://raw.githubusercontent.com/{GITHUB_REPO}/"
        f"{GITHUB_BRANCH}/{GITHUB_DATA_SUBDIR}/{CSV_NAME}"
    )
    print(f"Downloading from GitHub: {CSV_NAME}")
    urllib.request.urlretrieve(csv_url, csv_path)

# Index CSV rows by doc_id
meta_by_doc_id = {}
with open(csv_path, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        meta_by_doc_id[row["doc_id"]] = row

print(f"Loaded {len(meta_by_doc_id)} metadata rows from {csv_path}")

# Join each selected .md to its CSV row — one entry per document (no merge)
documents = []  # list of {"doc_id", "path", "text", "meta"}
missing = []

for path in selected_paths:
    doc_id = path.stem  # filename without .md
    meta = meta_by_doc_id.get(doc_id)
    if meta is None:
        missing.append(doc_id)
        continue
    text = path.read_text(encoding="utf-8", errors="ignore")
    documents.append({
        "doc_id": doc_id,
        "path": path,
        "text": text,
        "meta": meta,
    })

if missing:
    print(f"WARNING: no CSV row for {len(missing)} file(s): {missing[:5]}{'...' if len(missing) > 5 else ''}")

if not documents:
    raise RuntimeError("No documents joined. Check SELECTED_FILES / raw_data.csv.")

print(f"Joined {len(documents)} document(s) (per-doc, not merged)")
for d in documents:
    print(f"  - {d['doc_id']}: {len(d['text'])} chars")


Loaded 285 metadata rows from docs_from_github/raw_data.csv
Joined 271 document(s) (per-doc, not merged)
  - 0154c3ea-32a1-4eac-a40d-cc394c506a97: 5353 chars
  - 0167d8f6-b000-4304-bbfe-5c981ac1aa55: 13644 chars
  - 01795032-f63f-421e-8387-ddbc5df246c4: 5268 chars
  - 02359dc4-8ff4-4084-a694-8a516904e100: 5308 chars
  - 027d87b6-6cff-4e78-ad8f-e51761a424e6: 4766 chars
  - 035de21d-4241-4bb2-933c-4d5b0fbaae7c: 5238 chars
  - 03944a6e-edf7-477e-883e-7c9753b2685e: 4816 chars
  - 03a71673-a30f-42cb-a3e3-936f08d156d0: 5261 chars
  - 03ca845f-5a67-4c43-98f5-e4e7220a1d86: 5455 chars
  - 04bcd169-ce6a-4836-9dca-51f817dc1d70: 3013 chars
  - 05cb9a8a-cec4-4d84-961b-0d170560a0ea: 5455 chars
  - 06393c5f-3f0d-4365-827d-237c07f470ec: 5382 chars
  - 0647780e-3957-4813-b349-be510031a2c6: 5473 chars
  - 077739c9-0e0b-49b0-bad2-f7cc9fa3be20: 6202 chars
  - 077e48d9-daec-4a43-bec0-079d65d68207: 5256 chars
  - 07fb6de9-05d8-40ac-a633-704faef76ae7: 5353 chars
  - 081f3adf-0ed0-44b0-ad81-e6f3e1493bfd: 1285

In [5]:
# Inspect the first joined document (change index to peek at another)
doc = documents[0]
meta = doc["meta"]

print("doc_id:              ", doc["doc_id"])
print("markdown file:       ", doc["path"].name)
print("text length (chars): ", len(doc["text"]))
print()
print("CSV metadata:")
for key in [
    "doc_id",
    "base_url",
    "canonical_url",
    "crawl_dt",
    "doc_last_modified_dt",
    "content_type",
    "content_source_type",
    "scheme_type",
    "scheme_name",
    "lang",
    "doc_version",
    "is_active",
    "status",
    "crawl_depth",
    "normalized_url",
]:
    print(f"  {key:<22} {meta.get(key, '')}")

print()
print("text preview:")
print(doc["text"][:300].replace("\n", " "), "...")


doc_id:               0154c3ea-32a1-4eac-a40d-cc394c506a97
markdown file:        0154c3ea-32a1-4eac-a40d-cc394c506a97.md
text length (chars):  5353

CSV metadata:
  doc_id                 0154c3ea-32a1-4eac-a40d-cc394c506a97
  base_url               https://startup.telangana.gov.in
  canonical_url          https://startup.telangana.gov.in/locations/ag-hub-pjtsau-telangana/
  crawl_dt               2026-07-01T13:03:13.087174+00:00
  doc_last_modified_dt   2026-07-01T13:03:13.087174+00:00
  content_type           text/html; charset=UTF-8
  content_source_type    web
  scheme_type            HTTPS
  scheme_name            startup.telangana.gov.in
  lang                   en
  doc_version            1
  is_active              True
  status                 200
  crawl_depth            2
  normalized_url         https://startup.telangana.gov.in/locations/ag-hub-pjtsau-telangana

text preview:
Ag-Hub, PJTSAU, Telangana – Startup Telangana Department of ITE &C, Government of Telangana tsic@tel

### 6. Prepare all documents (ingest)

No single-doc experiment index. Strategies below loop over every entry in `documents` (from §3–§4).

Run **exactly one** strategy cell — each writes:
- `chunked_docs` — per-document `{doc_id, meta, chunks}`
- `chunks` — flat list of all chunk texts (for peek cells)


`SELECTED_FILES = []` in §3 → all markdown files. Next cell only verifies `documents` is ready.


In [13]:
# Ingest ALL joined documents (not a single experiment index)
assert documents, "Run §3–§4 first so documents is populated"

chunked_docs = []  # filled by the strategy cell you run next
chunks = []        # flat list across all docs (filled by strategy)

print(f"Ready to chunk {len(documents)} documents")
print(f"First doc_id: {documents[0]['doc_id']}")
print(f"Total characters: {sum(len(d['text']) for d in documents):,}")


Ready to chunk 271 documents
First doc_id: 0154c3ea-32a1-4eac-a40d-cc394c506a97
Total characters: 3,740,969


### 7. A shared token counter

Optional helper for peeking at chunk sizes.


In [8]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")
def n_tokens(text: str) -> int:
    return len(enc.encode(text))

###

- `VECTOR_DIM` — embedding output size  
- `MAX_SEQ_LENGTH` — max tokens the model reads (chunks larger than this get truncated when embedding)  
- `CHUNK_SIZE` / `OVERLAP` — set from `MAX_SEQ_LENGTH` for token strategies below


In [14]:
import os

# Optional: set via env var to avoid Colab HF_TOKEN vault warnings. Public
# models like all-MiniLM-L6-v2 usually work without it. Never hardcode here.
HF_TOKEN = os.getenv("HF_TOKEN", "")
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
"BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(EMBEDDING_MODEL)

VECTOR_DIM = embedder.get_sentence_embedding_dimension()
MAX_SEQ_LENGTH = embedder.max_seq_length

# Token chunking aligned to what the embedder can actually consume
CHUNK_SIZE = int(MAX_SEQ_LENGTH)
OVERLAP = max(32, CHUNK_SIZE // 8)  # ~12.5% overlap

print(f"EMBEDDING_MODEL: {EMBEDDING_MODEL}")
print(f"VECTOR_DIM:      {VECTOR_DIM}")
print(f"MAX_SEQ_LENGTH:  {MAX_SEQ_LENGTH}")
try:
    print(f"model.config.hidden_size: {embedder[0].auto_model.config.hidden_size}")
except Exception:
    pass
print(f"CHUNK_SIZE:      {CHUNK_SIZE}  (aligned to MAX_SEQ_LENGTH)")
print(f"OVERLAP:         {OVERLAP}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

EMBEDDING_MODEL: sentence-transformers/all-MiniLM-L6-v2
VECTOR_DIM:      384
MAX_SEQ_LENGTH:  256
model.config.hidden_size: 384
CHUNK_SIZE:      256  (aligned to MAX_SEQ_LENGTH)
OVERLAP:         32


/tmp/ipykernel_7211/3744015324.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  VECTOR_DIM = embedder.get_sentence_embedding_dimension()


### 9. Strategy — Token-based WITH overlap (all docs)

Uses §8 `CHUNK_SIZE` / `OVERLAP`. Sets `chunked_docs` + flat `chunks`.


In [15]:
# Chunk ALL documents — token WITH overlap (§8 CHUNK_SIZE / OVERLAP)
print(f"OVERLAP: {OVERLAP}  CHUNK_SIZE: {CHUNK_SIZE}")
print(f"Documents: {len(documents)}")

splitter = TokenTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=OVERLAP)
chunked_docs = []
chunks = []

for doc in documents:
    nodes = splitter.get_nodes_from_documents([Document(text=doc["text"])])
    doc_chunks = [n.text for n in nodes]
    chunked_docs.append({"doc_id": doc["doc_id"], "meta": doc["meta"], "chunks": doc_chunks})
    chunks.extend(doc_chunks)

print(f"Token WITH overlap -> {len(chunks)} chunks across {len(chunked_docs)} docs")
print(chunks[0][:512], "...")


OVERLAP: 32  CHUNK_SIZE: 256
Documents: 271
Token WITH overlap -> 3599 chunks across 271 docs
Ag-Hub, PJTSAU, Telangana – Startup Telangana Department of ITE &C, Government of Telangana tsic@telangana.gov.in About Vision Leadership Nodal Officer Nodal Team Dedicated team for Startups Nodal Department Know Your Ecosystem State Startup Network Startups State Startup Policy Startup Definition Startup Recognition Registration Manual Approval and Clearances Startup Dashboard Funding Funding and Incentives Public Procurement Investors Co Working Women Entrepreneurship Enablers Government Innovation Incuba ...


### 10. Strategy — Token-based WITHOUT overlap

Same `CHUNK_SIZE` as §8/§9, but `chunk_overlap=0`.

Sets `chunks`.


In [ ]:
# Chunk ALL documents — token WITHOUT overlap
splitter = TokenTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=0)
chunked_docs = []
chunks = []

for doc in documents:
    nodes = splitter.get_nodes_from_documents([Document(text=doc["text"])])
    doc_chunks = [n.text for n in nodes]
    chunked_docs.append({"doc_id": doc["doc_id"], "meta": doc["meta"], "chunks": doc_chunks})
    chunks.extend(doc_chunks)

print(f"Token WITHOUT overlap (chunk_size={CHUNK_SIZE}) -> {len(chunks)} chunks across {len(chunked_docs)} docs")
print(chunks[0][:200], "...")


### 11. Strategy — Sentence-based

chunk =  1 sentence / paragraph


In [11]:
# Chunk ALL documents — sentence-based
splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=150,
    paragraph_separator="\n\n",
    secondary_chunking_regex=r"[.!?]\s+",
)
chunked_docs = []
chunks = []

for doc in documents:
    nodes = splitter.get_nodes_from_documents([Document(text=doc["text"])])
    doc_chunks = [n.text for n in nodes]
    chunked_docs.append({"doc_id": doc["doc_id"], "meta": doc["meta"], "chunks": doc_chunks})
    chunks.extend(doc_chunks)

print(f"Sentence-based -> {len(chunks)} chunks across {len(chunked_docs)} docs")
print(chunks[0][:200], "...")


Sentence-based -> 5 chunks
Events – Startup Telangana Department of ITE &C, Government of Telangana tsic@telangana.gov.in About Vision Leadership Nodal Officer Nodal Team Dedicated team for Startups Nodal Department Know Your E ...


### 12. Strategy — Semantic

chunk = Split where meaning shifts (needs embeddings; slower).


In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from scipy.spatial.distance import cosine

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# Same model string as §8 (loads again for llama-index API used here)
embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL)
print(f"Semantic chunking using {EMBEDDING_MODEL}")
print(f"Documents: {len(documents)} (slow — embeds sentences per doc)")


def split_into_sentences(text, min_length=30):
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if len(s.strip()) >= min_length]


def semantic_chunk(text, distance_percentile=95):
    sentences = split_into_sentences(text)
    if len(sentences) < 2:
        return sentences
    embeddings = np.array(embed_model.get_text_embedding_batch(sentences, show_progress=False))
    distances = np.array([cosine(embeddings[i - 1], embeddings[i]) for i in range(1, len(sentences))])
    threshold = np.percentile(distances, distance_percentile)
    breakpoints = [i for i, d in enumerate(distances) if d >= threshold]
    chunks_out, start = [], 0
    for bp in breakpoints:
        chunks_out.append(" ".join(sentences[start:bp + 1]))
        start = bp + 1
    chunks_out.append(" ".join(sentences[start:]))
    return [c for c in chunks_out if c.strip()]


chunked_docs = []
chunks = []
for i, doc in enumerate(documents):
    doc_chunks = semantic_chunk(doc["text"])
    chunked_docs.append({"doc_id": doc["doc_id"], "meta": doc["meta"], "chunks": doc_chunks})
    chunks.extend(doc_chunks)
    if (i + 1) % 10 == 0 or i + 1 == len(documents):
        print(f"  {i + 1}/{len(documents)} docs …")

print(f"Semantic -> {len(chunks)} chunks across {len(chunked_docs)} docs")
print(chunks[0][:200], "...")


### 13. Peek chunks (whatever strategy ran)


In [16]:
assert chunked_docs, "Run one strategy cell first"
print(f"docs:   {len(chunked_docs)}")
print(f"chunks: {len(chunks)}")
print(f"first chunk tokens: {n_tokens(chunks[0])}")
print()
print(chunks[0][:400], "...")


docs:   271
chunks: 3599
first chunk tokens: 254

Ag-Hub, PJTSAU, Telangana – Startup Telangana Department of ITE &C, Government of Telangana tsic@telangana.gov.in About Vision Leadership Nodal Officer Nodal Team Dedicated team for Startups Nodal Department Know Your Ecosystem State Startup Network Startups State Startup Policy Startup Definition Startup Recognition Registration Manual Approval and Clearances Startup Dashboard Funding Funding and ...


In [17]:
# Build schema-aligned records for ALL docs (in-memory; text_vector added in Milvus ingest below)
import json
import uuid
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path

TEXT_MAX_LEN = 15000
RECORDS_FILE = Path("selected_chunks.json")


def to_epoch_ms(value) -> int:
    if value is None or value == "":
        return 0
    if isinstance(value, (int, float)):
        return int(value)
    s = str(value).strip()
    try:
        dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
    except ValueError:
        dt = parsedate_to_datetime(s)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)


def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"1", "true", "yes"}


assert chunked_docs, "Run one strategy cell first so chunked_docs is set"

records = []
for item in chunked_docs:
    meta = item["meta"]
    doc_id = item["doc_id"]
    for order, text in enumerate(item["chunks"]):
        text = (text or "")[:TEXT_MAX_LEN]
        content_type = (meta.get("content_type") or "").split(";")[0].strip()[:20]
        records.append({
            "chunk_id": str(uuid.uuid4()),
            "document_id": doc_id,
            "chunk_order": int(order),
            "base_url": meta.get("base_url", ""),
            "canonical_url": meta.get("canonical_url", ""),
            "crawl_date": to_epoch_ms(meta.get("crawl_dt")),
            "doc_last_modified": to_epoch_ms(meta.get("doc_last_modified_dt")),
            "content_type": content_type,
            "content_source_type": meta.get("content_source_type", ""),
            "scheme_type": meta.get("scheme_type", ""),
            "scheme_name": meta.get("scheme_name", ""),
            "language": meta.get("lang", ""),
            "text": text,
            "doc_version": str(meta.get("doc_version", "")),
            "is_active": as_bool(meta.get("is_active", True)),
        })

RECORDS_FILE.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {len(records)} records from {len(chunked_docs)} docs -> {RECORDS_FILE.resolve()}")
print("Keys:", list(records[0].keys()))
print("bytes:", RECORDS_FILE.stat().st_size)


Wrote 3599 records from 271 docs -> /content/selected_chunks.json
Keys: ['chunk_id', 'document_id', 'chunk_order', 'base_url', 'canonical_url', 'crawl_date', 'doc_last_modified', 'content_type', 'content_source_type', 'scheme_type', 'scheme_name', 'language', 'text', 'doc_version', 'is_active']
bytes: 6440895


### 14. Milvus ingest (schema already created)

Assumes `rag_chunks` exists on Zilliz (from `schema_design_notebook`).

Uses in-memory `records` from the previous cell (no JSON required for insert).
Run: one strategy → build records → connect → embed → insert.


In [19]:
# --- Connect to Zilliz (do not recreate schema) ---
import os
from pymilvus import MilvusClient

# Set these via environment variables (e.g. a local .env file) — do not hardcode.
ZILLIZ_URI = os.getenv("ZILLIZ_URI", "")
ZILLIZ_USER = os.getenv("ZILLIZ_USER", "")
ZILLIZ_PASSWORD = os.getenv("ZILLIZ_PASSWORD", "")

if not ZILLIZ_URI or not ZILLIZ_USER or not ZILLIZ_PASSWORD:
    raise ValueError("Set ZILLIZ_URI, ZILLIZ_USER, and ZILLIZ_PASSWORD as environment variables before running this cell")

COLLECTION_NAME = "rag_chunks"
client = MilvusClient(uri=ZILLIZ_URI, token=f"{ZILLIZ_USER}:{ZILLIZ_PASSWORD}")

assert client.has_collection(COLLECTION_NAME), (
    f"Missing collection {COLLECTION_NAME}. Create it in schema_design_notebook first."
)
print("Connected. Collection ready:", COLLECTION_NAME)
print("Collections:", client.list_collections())


Connected. Collection ready: rag_chunks
Collections: ['rag_chunks']


In [20]:
# --- Embed text → text_vector (must match collection dim, MiniLM = 384) ---
assert records, "Run the records cell above first"

# Reuse embedder from §8 if present; otherwise load MiniLM
try:
    embedder
    VECTOR_DIM
    EMBEDDING_MODEL
except NameError:
    from sentence_transformers import SentenceTransformer
    EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
    embedder = SentenceTransformer(EMBEDDING_MODEL)
    VECTOR_DIM = embedder.get_sentence_embedding_dimension()

print(f"EMBEDDING_MODEL={EMBEDDING_MODEL} VECTOR_DIM={VECTOR_DIM}")

texts = [r["text"] for r in records]
vectors = embedder.encode(texts, show_progress_bar=True)

for r, vec in zip(records, vectors):
    v = vec.tolist() if hasattr(vec, "tolist") else list(vec)
    if len(v) != VECTOR_DIM:
        raise ValueError(f"Embedding dim {len(v)} != {VECTOR_DIM}")
    r["text_vector"] = v

print(f"Embedded {len(records)} records; text_vector length={len(records[0]['text_vector'])}")


EMBEDDING_MODEL=sentence-transformers/all-MiniLM-L6-v2 VECTOR_DIM=384


Batches:   0%|          | 0/113 [00:00<?, ?it/s]

Embedded 3599 records; text_vector length=384


In [21]:
# --- Insert into existing rag_chunks collection ---
res = client.insert(collection_name=COLLECTION_NAME, data=records)
print("Insert result:", res)
print(f"Inserted {len(records)} rows into {COLLECTION_NAME}")


Insert result: {'insert_count': 3599, 'ids': ['54366046-01b2-4c38-8a50-c2f3575ce5f9', '7dca14a7-dd86-4f7b-b474-b5157e71634e', '148d5e40-81f2-49a9-8981-aa90c8eafc3b', '06320701-38e5-451e-9ec0-998a880eed38', '47d5f267-3885-4b4c-8f17-70bf1f571323', 'b5f03208-836a-40fb-bf31-502d95bf9d2d', '182ced1a-ff51-4381-ad71-e10e83de0ff0', '075da6dd-c1a9-4923-a0e4-2d6e1a222c21', '026a4cb2-127a-4e76-9640-8f5f9a848553', '681b0453-3747-450d-ae39-66987f192537', '95c3e9f1-8a00-4940-8757-1eda815b27b6', 'de321098-5c9a-45d0-8363-03b830382fd7', 'bc4e569a-48b8-4bae-82f1-28aae06628aa', '92b13d3e-8e41-43c0-bb29-ce0c0f3f251b', 'd9035446-a324-4fee-8635-4fd07331e534', '059def8b-d9d4-4bc8-a6b5-c007945236a7', '953b1121-c4cc-47d6-a85f-b73d3e1a352d', '6dcb6f95-09f6-4aea-9295-7e41818eb77f', '4c8fb86b-1e61-4aee-86cb-e3146f88f174', '5c78797a-9488-4a59-89e9-8212c7ce4720', '942610b3-918e-4eaf-b1da-c9ca27eb62ca', '63c58de5-0bbd-4eb8-bf06-eefd63ade7b9', 'd24deeb2-c7d0-4736-a81d-3a08dd73f4e4', '68cd491c-cc93-45f9-a9dd-6524f1341

In [24]:
# --- Peek a few rows ---
# Needs vector index first (create in schema_design_notebook §12), then load
client.load_collection(COLLECTION_NAME)
print("Collection loaded:", COLLECTION_NAME)

rows = client.query(
    collection_name=COLLECTION_NAME,
    filter="chunk_order >= 0",
    output_fields=["chunk_id", "document_id", "chunk_order", "text"],
    limit=min(5, len(records)),
)
print(f"Query returned {len(rows)} rows")
for row in rows:
    text = (row.get("text") or "")[:120].replace("\n", " ")
    print(f"- {row.get('document_id')} #{row.get('chunk_order')}: {text}...")


Collection loaded: rag_chunks
Query returned 5 rows
- 36711d02-bf94-4736-900f-ddb1941d0e7f #41: and solutions. TSIC offers various programs and initiatives, including incubation programs, mentoring, funding, and netw...
- 62811ed4-d580-4d6f-95d0-497af1a9a77c #149: Malla Reddy College of Engineering and Technology 73 Mylavarapu Ganesh Kumar Malla Reddy College of Engineering and Tech...
- f5323729-a67a-43ed-8195-cfaebb07db17 #0: Signin | Startup Telangana Please read the instructions First Name should be between 2-50 characters without any numbers...
- 9139e4fd-fa60-4e46-9ef4-8ed948e695ea #4: Khairatabad, Hyderabad, Telangana 500022. tsic@telangana.gov.in +91 9100678543 QUICK ACCESS Vision Know Your Ecosystem P...
- 78a440b7-a2bf-43f0-ad09-f20db0d09eb5 #5: Telangana Secretariat Rd, Khairatabad, Hyderabad, Telangana 500022. tsic@telangana.gov.in +91 9100678543 QUICK ACCESS Vi...


### 15. Retrieve — hardcode a query, get top-k chunks

Requires: connected client, loaded collection, vector index on `text_vector` (schema §12), and the same embedder as ingest (MiniLM).


In [29]:
# --- Vector search: edit QUERY, run cell ---
QUERY = "What funding or schemes are available for social cause by women entrepreneur?"
TOP_K = 5

# Same model as ingest (§8 / embed cell)
try:
    embedder
except NameError:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

client.load_collection(COLLECTION_NAME)

query_vec = embedder.encode([QUERY])[0]
query_vec = query_vec.tolist() if hasattr(query_vec, "tolist") else list(query_vec)

results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vec],
    anns_field="text_vector",
    limit=TOP_K,
    search_params={"metric_type": "COSINE", "params": {}},
    output_fields=["chunk_id", "document_id", "chunk_order", "canonical_url", "text"],
)

print(f"Query: {QUERY}\n")
for hit in results[0]:
    ent = hit["entity"]
    doc_id = ent.get("document_id") or ""
    score = hit["distance"]  # COSINE similarity (higher = more similar)
    print(f"source: {doc_id}.md")
    print(f"score:  {score:.4f}")
    print(ent.get("text", ""))
    print("---")


Query: What funding or schemes are available for social cause by women entrepreneur?

source: 30b2ee04-40f5-499f-8086-54ef3e8f7e77.md
score:  0.5847
“Our economy and society reach its full potential only when women are empowered.” WE Hub is India’s first state-led nodal agency to promote and foster women entrepreneurship by incubating, mentoring, and creating a global collaborative network for women-led enterprises from both rural and urban landscapes and providing them access to technical, financial, governmental and policy support required. Any woman entrepreneur, at any stage of her entrepreneurial journey is welcome at WE Hub irrespective of her sector of operation and education, financial or demographic background. WE Hub’s primary role is to create an enabling ecosystem around a woman entrepreneur to facilitate the growth of her enterprise. Service Offerings Incubation Government Support Ecosystem Building Mentoring Compliances & Tax Assistance Empower women entrepreneurs from va